# WTMC2021 CICIDS2017 EDA and Data Cleaning

**Project:** Network Anomaly Detection and Predictive Intrusion Monitoring Platform  
**Dataset:** WTMC2021 improved/regenerated CICIDS2017 flow-based dataset  
**Scope:** Exploratory data analysis and reproducible data preparation only

This notebook does not train machine-learning models or apply resampling, scaling, feature selection, or SMOTE.

## Objective

Inspect the actual local dataset structure, understand its labels and quality, document potentially unnecessary identifier columns, and create a cleaned dataset while preserving the original attack labels. Every reported count below is calculated after the dataset is loaded.

## Dataset Source

The raw data is the WTMC2021 improved/regenerated version of the CICIDS2017 flow-based dataset. Raw files are kept local because of their size. See [`data/README.md`](../data/README.md) for placement instructions.

## Import Required Libraries

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 100)
sns.set_theme(style='whitegrid', context='notebook')
RANDOM_STATE = 42

## Load WTMC2021 CICIDS2017 Dataset

In [ ]:
# Change this path when the raw dataset is stored somewhere else.
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / 'CICIDS2017_improved'
CSV_FILES = sorted(DATA_PATH.glob('*.csv'))
SAMPLE_ROWS_PER_FILE = None  # Set an integer for a quick exploratory sample; keep None for all rows.

if not CSV_FILES:
    raise FileNotFoundError(f'No CSV files found in {DATA_PATH.resolve()}. Update DATA_PATH.')

frames = []
for csv_file in CSV_FILES:
    read_kwargs = {} if SAMPLE_ROWS_PER_FILE is None else {'nrows': SAMPLE_ROWS_PER_FILE}
    frame = pd.read_csv(csv_file, low_memory=False, **read_kwargs)
    frame['Source_File'] = csv_file.name
    frames.append(frame)

data = pd.concat(frames, ignore_index=True)
print(f'Loaded {len(CSV_FILES)} file(s): {[file.name for file in CSV_FILES]}')
print(f'Raw rows loaded: {len(data):,}')

## Display Dataset Shape

In [ ]:
print(f'Rows: {data.shape[0]:,}')
print(f'Columns: {data.shape[1]:,}')

## Display Column Names

In [ ]:
for number, column in enumerate(data.columns, start=1):
    print(f'{number:>3}: {column!r}')

## Inspect Data Types

In [ ]:
display(pd.DataFrame({'dtype': data.dtypes.astype(str), 'non_null': data.notna().sum(), 'unique': data.nunique(dropna=True)}))

## Check Missing Values

In [ ]:
missing_counts = data.isna().sum().sort_values(ascending=False)
missing_summary = missing_counts[missing_counts.gt(0)].to_frame('missing_values')
display(missing_summary)
print(f'Rows containing at least one missing value: {data.isna().any(axis=1).sum():,}')

if not missing_summary.empty:
    plt.figure(figsize=(10, 5))
    sns.barplot(x=missing_summary.head(20).values.ravel(), y=missing_summary.head(20).index, color='#d95f02')
    plt.title('Missing Values by Column (Top 20)')
    plt.xlabel('Missing value count')
    plt.ylabel('Column')
    plt.tight_layout()
    plt.show()

## Check Infinite Values

In [ ]:
numeric_data = data.select_dtypes(include=np.number)
infinite_counts = np.isinf(numeric_data).sum().sort_values(ascending=False)
infinite_summary = infinite_counts[infinite_counts.gt(0)].to_frame('infinite_values')
display(infinite_summary)
print(f'Rows containing at least one infinite numeric value: {np.isinf(numeric_data).any(axis=1).sum():,}')

## Check Duplicate Rows

In [ ]:
duplicate_count = int(data.duplicated().sum())
print(f'Duplicate rows: {duplicate_count:,}')

## Analyze Target/Label Column

The target is inferred from actual column names and values. No fixed `Label` column name is assumed.

In [ ]:
label_name_pattern = re.compile(r'(label|class|attack|target|category|outcome|result)', re.IGNORECASE)
label_candidates = [column for column in data.columns if label_name_pattern.search(str(column))]
print('Likely label/target columns based on names:')
print(label_candidates if label_candidates else 'None found by name; inspect object/category columns below.')

value_candidates = [column for column in data.columns if data[column].dtype == 'object' or str(data[column].dtype) == 'category']
print('Categorical/text columns for manual target review:')
print(value_candidates)

TARGET_COLUMN = label_candidates[0] if label_candidates else (value_candidates[-1] if value_candidates else None)
if TARGET_COLUMN is None:
    raise ValueError('No likely target column was found. Set TARGET_COLUMN manually after inspecting the displayed columns.')
print(f'Selected target column for analysis: {TARGET_COLUMN!r}')
display(data[TARGET_COLUMN].value_counts(dropna=False).head(30).to_frame('count'))

## Analyze Benign vs Attack Distribution

In [ ]:
label_text = data[TARGET_COLUMN].astype('string').str.strip()
benign_mask = label_text.str.casefold().eq('benign')
benign_attack = pd.Series(np.where(benign_mask, 'Benign', 'Attack'), index=data.index, name='Traffic_Type')
distribution = benign_attack.value_counts().rename_axis('Traffic_Type').to_frame('count')
distribution['percentage'] = distribution['count'].div(len(data)).mul(100)
display(distribution)

plt.figure(figsize=(6, 4))
sns.countplot(x=benign_attack, order=['Benign', 'Attack'], color='#1b9e77')
plt.title('Benign vs Attack Distribution')
plt.xlabel('Traffic type')
plt.ylabel('Rows')
plt.tight_layout()
plt.show()

## Analyze `Attempted` Labels if Present

In [ ]:
attempted_mask = label_text.str.contains('attempt', case=False, na=False)
attempted_count = int(attempted_mask.sum())
print(f'Labels containing Attempted: {attempted_count:,}')
if attempted_count:
    display(data.loc[attempted_mask, TARGET_COLUMN].value_counts().to_frame('count'))
else:
    print('No Attempted label was found in the selected target column.')
print('These labels are preserved; no binary conversion is written back to the original target.')

## Analyze Individual Attack Classes

In [ ]:
class_counts = data[TARGET_COLUMN].value_counts(dropna=False).rename_axis('Original_Label').to_frame('count')
class_counts['percentage'] = class_counts['count'].div(len(data)).mul(100)
display(class_counts)

plt.figure(figsize=(10, max(4, min(12, len(class_counts) * 0.35))))
sns.barplot(data=class_counts.reset_index(), x='count', y='Original_Label', color='#7570b3')
plt.title('Individual Original Label Distribution')
plt.xlabel('Rows')
plt.ylabel('Original label')
plt.tight_layout()
plt.show()

## Statistical Summary of Numerical Features

In [ ]:
display(data.select_dtypes(include=np.number).describe().T)

## Feature Distributions

In [ ]:
numeric_columns = data.select_dtypes(include=np.number).columns.tolist()
plot_columns = [column for column in numeric_columns if column != TARGET_COLUMN][:6]
print(f'Selected numerical columns for visualization: {plot_columns}')
if plot_columns:
    data[plot_columns].hist(bins=40, figsize=(14, 10), color='#e7298a')
    plt.suptitle('Selected Numerical Feature Distributions')
    plt.tight_layout()
    plt.show()
else:
    print('No numerical columns available for distribution plots.')

## Identify Potential Identifier/Unnecessary Columns

Identifiers are inspected and reported, but they are not removed automatically.

In [ ]:
identifier_pattern = re.compile(r'(flow.?id|source.?ip|destination.?ip|timestamp|time|^id$)', re.IGNORECASE)
identifier_columns = [column for column in data.columns if identifier_pattern.search(str(column))]
identifier_review = pd.DataFrame({
    'column': identifier_columns,
    'dtype': [str(data[column].dtype) for column in identifier_columns],
    'unique_values': [data[column].nunique(dropna=True) for column in identifier_columns],
    'unique_fraction': [data[column].nunique(dropna=True) / len(data) for column in identifier_columns],
})
display(identifier_review)
print('Review guidance: Flow IDs, IP addresses, and timestamps may leak identity or capture-time information, but removal depends on the planned deployment and feature policy.')

## Document Reasons for Removing Any Columns

No columns are removed by default. After reviewing the table above and the project threat model, list only deliberate decisions here.

- `Flow ID`, source/destination IP, and timestamp are retained in this preparation output unless a documented modeling policy removes them later.
- The helper column `Source_File` records provenance for the concatenated local files and can be removed before modeling if file provenance is not required.
- The original target column is preserved exactly; `Traffic_Type` is analysis-only and is not used to overwrite the original labels.

## Clean Invalid/Missing/Infinite Values

In [ ]:
cleaned = data.copy()
cleaned['Original_Label'] = cleaned[TARGET_COLUMN].copy()
rows_before_cleaning = len(cleaned)

numeric_columns = cleaned.select_dtypes(include=np.number).columns
infinite_row_mask = np.isinf(cleaned[numeric_columns]).any(axis=1) if len(numeric_columns) else pd.Series(False, index=cleaned.index)
missing_row_mask = cleaned.isna().any(axis=1)
invalid_row_mask = missing_row_mask | infinite_row_mask
invalid_rows_removed = int(invalid_row_mask.sum())
cleaned = cleaned.loc[~invalid_row_mask].copy()

print(f'Rows before cleaning: {rows_before_cleaning:,}')
print(f'Rows with missing values removed: {int(missing_row_mask.sum()):,}')
print(f'Rows with infinite values removed: {int(infinite_row_mask.sum()):,}')
print(f'Rows removed by invalid-value operation (overlap counted once): {invalid_rows_removed:,}')
print(f'Rows after invalid-value cleaning: {len(cleaned):,}')

## Handle Duplicate Rows if Appropriate

In [ ]:
duplicates_before_removal = int(cleaned.duplicated().sum())
cleaned = cleaned.drop_duplicates().copy()
print(f'Rows before duplicate handling: {len(cleaned) + duplicates_before_removal:,}')
print(f'Duplicate rows removed: {duplicates_before_removal:,}')
print(f'Rows after duplicate handling: {len(cleaned):,}')
print(f'Overall rows removed: {rows_before_cleaning - len(cleaned):,}')

## Verify the Cleaned Dataset

In [ ]:
print(f'Cleaned shape: {cleaned.shape}')
print(f'Remaining missing values: {int(cleaned.isna().sum().sum()):,}')
cleaned_numeric = cleaned.select_dtypes(include=np.number)
print(f'Remaining infinite values: {int(np.isinf(cleaned_numeric).sum().sum()) if len(cleaned_numeric.columns) else 0:,}')
print(f'Remaining duplicate rows: {int(cleaned.duplicated().sum()):,}')
print(f'Original target column preserved: {TARGET_COLUMN in cleaned.columns}')
print('Original label copy present:', 'Original_Label' in cleaned.columns)

## Compare Dataset Before vs After Cleaning

In [ ]:
comparison = pd.DataFrame({
    'Before cleaning': [len(data), data.shape[1], int(data.isna().sum().sum()), int(np.isinf(data.select_dtypes(include=np.number)).sum().sum()), int(data.duplicated().sum())],
    'After cleaning': [len(cleaned), cleaned.shape[1], int(cleaned.isna().sum().sum()), int(np.isinf(cleaned.select_dtypes(include=np.number)).sum().sum()), int(cleaned.duplicated().sum())]
}, index=['Rows', 'Columns', 'Missing cells', 'Infinite numeric cells', 'Duplicate rows'])
display(comparison)
print(f'Rows before -> after: {len(data):,} -> {len(cleaned):,}')

## Save the Cleaned Dataset

In [ ]:
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'WTMC2021_CICIDS2017_cleaned.csv'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
cleaned.to_csv(OUTPUT_PATH, index=False)
print(f'Saved {len(cleaned):,} cleaned rows to {OUTPUT_PATH.resolve()}')

## Final Data Quality Summary

In [ ]:
print('Data quality summary')
print('--------------------')
print(f'Input rows: {len(data):,}')
print(f'Output rows: {len(cleaned):,}')
print(f'Rows removed: {len(data) - len(cleaned):,}')
print(f'Target column analyzed: {TARGET_COLUMN}')
print('Original labels preserved:', TARGET_COLUMN in cleaned.columns and 'Original_Label' in cleaned.columns)
print(f'Missing cells after cleaning: {int(cleaned.isna().sum().sum()):,}')
print(f'Infinite numeric cells after cleaning: {int(np.isinf(cleaned.select_dtypes(include=np.number)).sum().sum()) if len(cleaned.select_dtypes(include=np.number).columns) else 0:,}')
print('No ML model training, resampling, scaling, or feature selection was performed.')